In [2]:
import os
import sys
import json
import numpy as np
import pandas as pd
import networkx as nx

import re
import pickle as pkl

In [3]:
# get directory list 
def absoluteFilePaths(directory):
    for dirpath,_,filenames in os.walk(directory):
        for f in filenames:
            if bool(re.search('.json', f)):
                myFiles.append(os.path.abspath(os.path.join(dirpath, f)))

myFiles = []
absoluteFilePaths("/mnt/raid0_ssd_8tb/Bibek/NCR/ecar/evaluation/23Sep19/AIA-201-225/")
myFiles = sorted(myFiles)
for file in myFiles:
    print(file)

/mnt/raid0_ssd_8tb/Bibek/NCR/ecar/evaluation/23Sep19/AIA-201-225/AIA-201-225.ecar-2019-12-08T11-05-10.046.json
/mnt/raid0_ssd_8tb/Bibek/NCR/ecar/evaluation/23Sep19/AIA-201-225/AIA-201-225.ecar-last.json


In [ ]:
events = set()
#pid2actorId = {}
# The graph need to encode the absolute direction of information flow 
# OBJECT: HOST
    # CASE: START ==> 
    
# OBJECT: USER_SESSION
    # CASE: INTERACTIVE ==>
    # CASE: GRANT ==>
    # CASE: LOGOUT ==> 
    # CASE: UNLOCK ==> 
    # CASE: REMOTE ==>
    # CASE: LOGIN ==>

# OBJECT: FILE
    # CASE: READ ==> Object --> Process
    # CASE: CREATE ==> Process --> Object
    # CASE: WRITE ==> Process --> Object
    # CASE: DELETE ==>
    # CASE: RENAME ==>
    # CASE: MODIFY ==> 

# OBJECT: PROCESS
    # CASE: CREATE ==> Old_process --> new_process
    # CASE: OPEN  ==> Old_process--> new_process
    # CASE: TERMINATE ==> Old_process --> new_process

# OBJECT: THREAD
    # CASE: CREATE ==>
    # CASE: REMOTE_CREATE ==> 
    # CASE: TERMINATE ==> 
    
# TASK: 
    # CASE: CREATE ==>
    # CASE: DELETE ==>
    # CASE: MODIFY ==>
    # CASE: START ==>

# OBJECT: REGISTRY
    # CASE: ADD ==> Process --> Registry
    # CASE: EDIT ==> Process --> Registry
    # CASE: REMOVE ==> Process --> Registry
    
# OBJECT: NETFLOW
    # CASE: OPEN ==> HOST --> d_Host
    # CASE: MESSAGE ==> HOST --> d_HOST
    # CASE: START ==> HOST --> d_HOST
    
# OBJECT: MODULE
    # CASE: LOAD ==> Module --> Process
    
# OBJECT: SHELL
    # CASE: COMMAND ==> Process --> Command

# first pass scan the nodes, and add them
# Create the process treee
# first pass scan the nodes, and add them
# Create the process treee
for fileName in myFiles:
    with open(fileName) as logFile:
        lCount = 0
        for line in logFile:
            line = line.strip()
            try:
                y = json.loads(line)
            except:
                print(line)

            # now get the subject and objects 
            objectType = y["object"]
            actionType = y["action"]
            events.add((objectType, actionType))

In [ ]:
objects = set()
for event in events:
    objects.add(event[0])
print(objects)

In [ ]:
for event in events:
    if event[0] == 'FILE':
        print(event)

In [ ]:
for event in events:
    if event[0] == 'HOST':
        print(event)

In [ ]:
for event in events:
    if event[0] == 'USER_SESSION':
        print(event)

In [ ]:
for event in events:
    if event[0] == 'PROCESS':
        print(event)

In [ ]:
for event in events:
    if event[0] == 'TASK':
        print(event)

In [ ]:
for event in events:
    if event[0] == 'REGISTRY':
        print(event)

In [ ]:
for event in events:
    if event[0] == 'THREAD':
        print(event)

In [ ]:
for event in events:
    if event[0] == 'FLOW':
        print(event)

In [ ]:
for event in events:
    if event[0] == 'SHELL':
        print(event)

In [ ]:
for event in events:
    if event[0] == 'MODULE':
        print(event)

In [1]:
for i in range(25):
    print(i)

0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24


In [4]:
graphs = []
for hostID in range(1, 26):
    G = nx.MultiDiGraph()
    graphs.append(G)
#    myHostname = "SysClient0{}.systemia.com".format(hostID)
#    print(myHostname)

In [5]:
# first pass scan the nodes, and add them
# Create the process treee
for fileName in myFiles:
    with open(fileName) as logFile:
        lCount = 0
        for line in logFile:
            line = line.strip()
            try:
                y = json.loads(line)
            except:
                print(line)
            
            # get the hostname and select correct graph
            idx = int(y["hostname"][10:13]) - 201
            
            G = graphs[idx]
            # now get the subject and objects 
            actorId = y["actorID"]
            pid = y["pid"]
            ppid = y["ppid"]
            
            objectId = y["objectID"]
            objectType = y["object"]
            
            G.add_node(actorId)
            G.add_node(objectId)
            
            G.nodes[actorId]["pid"] = pid
            G.nodes[actorId]["ppid"] = ppid
            G.nodes[actorId]["oType"] = "PROCESS"
            G.nodes[actorId]["principal"] = y["principal"]
            G.nodes[actorId]["hostname"] = y["hostname"]

            G.nodes[objectId]["oType"] = objectType

            properties = y["properties"]
            properties["action"] = y["action"]
            properties["timestamp"] = y["timestamp"]
            
            if y["object"] == "FILE" and y["action"] == "READ":
                G.add_edge(y["objectID"], y["actorID"], attributes = properties)
            elif y["object"] == "MODULE" and y["action"] == "LOAD":
                G.add_edge(y["objectID"], y["actorID"], attributes = properties)
            elif y["object"] == "FLOW" and y["properties"]["direction"] == "inbound":
                G.add_edge(y["objectID"], y["actorID"], attributes = properties)
            else:
                G.add_edge(y["actorID"], y["objectID"], attributes = properties)
                #processTree.add_edge(ppid, pid)
            

In [6]:
G = graphs[0]
for node in G.nodes:
    print(G.nodes[node])
  #  print(G[node])
    break
#print(G.adj["3fb3043-fa23-44f9-9ecd-03185550fb63")
for edge in G.edges:
    print(edge[0], edge[1], edge[2])
    print(G.edges[edge]['attributes'])
    break

{'pid': 2236, 'ppid': 1472, 'oType': 'PROCESS', 'principal': 'NT AUTHORITY\\SYSTEM', 'hostname': 'SysClient0201.systemia.com'}
2f016449-765c-4896-bfca-5be85b4da8aa e4f088fd-0372-43f0-bbae-da4d39450ee9 0
{'acuity_level': '1', 'command_line': 'C:\\Windows\\system32\\cmd.exe /c "tasklist /fi "IMAGENAME eq mantra.exe""', 'image_path': '\\Device\\HarddiskVolume1\\Windows\\system32\\cmd.exe', 'parent_image_path': '\\Device\\HarddiskVolume1\\Python27\\python.exe', 'sid': 'S-1-5-18', 'user': 'NT AUTHORITY\\SYSTEM', 'action': 'CREATE', 'timestamp': '2019-09-23T09:50:49.859-04:00'}


In [7]:
!rm /mnt/raid0_ssd_8tb/Bibek/NCR/ecar/evaluation/23Sep19/AIA-201-225/graph_2*

In [8]:
!ls /mnt/raid0_ssd_8tb/Bibek/NCR/ecar/evaluation/23Sep19/AIA-201-225/

AIA-201-225.ecar-2019-12-08T11-05-10.046.json  AIA-201-225.ecar-last.json


In [9]:
for i, G in enumerate(graphs):
    dir_string = "/mnt/raid0_ssd_8tb/Bibek/NCR/ecar/evaluation/23Sep19/AIA-201-225/graph_{}".format(i+201)
    print(dir_string)
    nx.write_gpickle(G, dir_string)

/mnt/raid0_ssd_8tb/Bibek/NCR/ecar/evaluation/23Sep19/AIA-201-225/graph_201
/mnt/raid0_ssd_8tb/Bibek/NCR/ecar/evaluation/23Sep19/AIA-201-225/graph_202
/mnt/raid0_ssd_8tb/Bibek/NCR/ecar/evaluation/23Sep19/AIA-201-225/graph_203
/mnt/raid0_ssd_8tb/Bibek/NCR/ecar/evaluation/23Sep19/AIA-201-225/graph_204
/mnt/raid0_ssd_8tb/Bibek/NCR/ecar/evaluation/23Sep19/AIA-201-225/graph_205
/mnt/raid0_ssd_8tb/Bibek/NCR/ecar/evaluation/23Sep19/AIA-201-225/graph_206
/mnt/raid0_ssd_8tb/Bibek/NCR/ecar/evaluation/23Sep19/AIA-201-225/graph_207
/mnt/raid0_ssd_8tb/Bibek/NCR/ecar/evaluation/23Sep19/AIA-201-225/graph_208
/mnt/raid0_ssd_8tb/Bibek/NCR/ecar/evaluation/23Sep19/AIA-201-225/graph_209
/mnt/raid0_ssd_8tb/Bibek/NCR/ecar/evaluation/23Sep19/AIA-201-225/graph_210
/mnt/raid0_ssd_8tb/Bibek/NCR/ecar/evaluation/23Sep19/AIA-201-225/graph_211
/mnt/raid0_ssd_8tb/Bibek/NCR/ecar/evaluation/23Sep19/AIA-201-225/graph_212
/mnt/raid0_ssd_8tb/Bibek/NCR/ecar/evaluation/23Sep19/AIA-201-225/graph_213
/mnt/raid0_ssd_8tb/Bibek/

In [ ]:
graphs.clear()